# Inter-Annotator Agreement Analysis

This notebook analyzes the agreement between different annotators on the misinformation dataset. It uses the functions defined in `Interannotator_Analysis.py` to calculate and display agreement scores.


In [8]:
# Add the 'src' directory to the system path to import our script
import sys
import os
# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the directory containing the script to the Python path
sys.path.append(notebook_dir)

# Import the functions from your script
from Interannotator_Analysis import get_dataframe, calculate_agreement, calculate_pairwise_agreement

# Import pandas for displaying DataFrames
import pandas as pd

# Configure pandas to display all columns
pd.set_option('display.max_columns', None)

print("Setup complete. Functions from Interannotator_Analysis.py are ready to use.")

Setup complete. Functions from Interannotator_Analysis.py are ready to use.


### 2. Load and Preprocess Data

Using `get_dataframe()` function from the Interannotator_Analysis script. This function handles finding all the `*annotations_cleaned.json` files, loading them, and concatenating them into a single, cleaned pandas DataFrame.

In [9]:
# Load the data
df = get_dataframe()

print("DataFrame loaded successfully.")
print(f"Total annotations loaded: {len(df)}")
print(f"Number of unique annotators: {df['Annotator'].nunique()}")
print(f"Number of unique items: {df['ID'].nunique()}")

# Display the first few rows of the loaded data
df.head()

Found 6 cleaned annotation files.
All cleaned files have been loaded and combined.
DataFrame loaded successfully.
Total annotations loaded: 2403
Number of unique annotators: 6
Number of unique items: 1000


,ID,Text,all_caps,exclamation_marks,hedging,adjectives,unk,Annotator
0,NaN,"No Food, No FEMA: Hurricane Michael’s Survivor...",[],[],[],[],[],Rachelle
1,1.0,Would-be looter in Hurricane Michael-ravaged F...,[],[!],"[Would, would]",[],[],Rachelle
2,2.0,His argument is correct. Hurricane Rita killed...,[],[],[],[],[],Rachelle
3,3.0,im praying for all of my friends down in the C...,[],[],[],[],[],Rachelle
4,4.0,To all my Texas streamers: PLEASE be safe if t...,[PLEASE],[!],[],[],[],Rachelle


### 3. Calculate Fleiss' Kappa (Overall Agreement)

Using the `calculate_agreement()` function from the Interannotator_Analysis.py file to compute Fleiss' Kappa. This metric is suitable for measuring agreement among multiple raters (more than two). Here, we are computing Fleiss's Kappa for four human annotators and 1 AI. The function automatically handles the calculation and prints a formatted table of the results.

In [10]:
# Calculate Fleiss' Kappa (including Gemini)
fleiss_all_df = calculate_agreement(df, include_gemini=True)

# Calculate Fleiss' Kappa (human only)
fleiss_human_df = calculate_agreement(df, include_gemini=False)


--- Fleiss' Kappa (Overall Agreement - Including Gemini) ---
                  Fleiss' Kappa       Agreement  Items
Feature                                               
all_caps                  0.504        Moderate   1001
exclamation_marks         0.844  Almost Perfect   1001
hedging                   0.489        Moderate   1001
adjectives                0.161          Slight   1001
unk                       0.525        Moderate   1001

--- Fleiss' Kappa (Overall Agreement - Excluding Gemini) ---
                  Fleiss' Kappa       Agreement  Items
Feature                                               
all_caps                  0.702     Substantial    999
exclamation_marks         0.808  Almost Perfect    999
hedging                   0.546        Moderate    999
adjectives                0.486        Moderate    999
unk                       0.555        Moderate    999


### 4. Calculate Cohen's Kappa (Pairwise Agreement)

Next, we compute Cohen's Kappa for every pair of annotators using the `calculate_pairwise_agreement()` function. This gives us a more detailed view of agreement between specific individuals.

In [11]:
# Calculate and display pairwise Cohen's Kappa
pairwise_results_df = calculate_pairwise_agreement(df)


--- Cohen's Kappa (Pairwise Agreement) ---
                   Gemini & Jasmine  Gemini & Jennifer  Gemini & Nicole  Gemini & Rachelle  Gemini & Shiao-li  Jasmine & Jennifer  Jasmine & Nicole  Jasmine & Rachelle  Jasmine & Shiao-li  Jennifer & Nicole  Jennifer & Rachelle  Jennifer & Shiao-li  Nicole & Rachelle  Nicole & Shiao-li  Rachelle & Shiao-li
Feature                                                                                                                                                                                                                                                                                                            
all_caps                      0.400              0.219            0.412              0.284              0.219               0.740             0.717               0.439               0.658              0.768                0.563                0.795              0.692              0.657                0.708
exclamation_marks             0.

### 5. Visualize Agreement Scores

The bar chart below shows a comparison between the Fleiss' Kappa scores across different features.

#### Fleiss' Kappa Scores by Feature

In [12]:
import altair as alt

def create_agreement_chart(results_df, title):
    if results_df is None or results_df.empty:
        return None
        
    # Prepare data: reset index and standardize column name for Altair
    # (Altair can fail on column names containing apostrophes)
    plot_df = results_df.reset_index()
    plot_df.columns = [c.replace("Fleiss' ", "Fleiss ") for c in plot_df.columns]
    
    # Create the chart
    bars = alt.Chart(plot_df).mark_bar().encode(
        x=alt.X('Feature:N', title='Annotation Feature', sort=None, axis=alt.Axis(labelAngle=-45)),
        y=alt.Y("Fleiss Kappa:Q", title="Fleiss' Kappa Score", scale=alt.Scale(domain=[0, 1])),
        color=alt.Color('Agreement:N', title='Agreement Level', 
                        sort=['Poor', 'Slight', 'Fair', 'Moderate', 'Substantial', 'Almost Perfect']),
        tooltip=['Feature', alt.Tooltip("Fleiss Kappa:Q", format='.3f'), 'Agreement', 'Items']
    )
    
    # Add text labels on top of bars
    text = bars.mark_text(
        align='center',
        baseline='bottom',
        dy=-5
    ).encode(
        text=alt.Text("Fleiss Kappa:Q", format='.3f')
    )
    
    return (bars + text).properties(
        title=title,
        width=500,
        height=250
    )

# Create both charts
chart_all = create_agreement_chart(fleiss_all_df, "Fleiss' Kappa: Human + AI")
chart_human = create_agreement_chart(fleiss_human_df, "Fleiss' Kappa: Human Only")

# Display them stacked vertically
if chart_all and chart_human:
    display(alt.vconcat(chart_all, chart_human))
elif chart_all:
    display(chart_all)
else:
    print("No agreement results available to plot.")

alt.VConcatChart(...)

#### Pairwise Cohen's Kappa Scores

For the pairwise results, the following heatmap visualizes the agreement matrix between all pairs of annotators for each feature.


In [13]:
if 'pairwise_results_df' in locals() and not pairwise_results_df.empty:
    # Melt the data: Features are index, columns are Annotator Pairs
    df_melted = pairwise_results_df.reset_index().melt(
        id_vars='Feature', 
        var_name='Annotator Pair', 
        value_name="Kappa"
    )
    
    # Create the heatmap
    heatmap = alt.Chart(df_melted).mark_rect().encode(
        x=alt.X('Annotator Pair:N', title='Annotator Pairs', axis=alt.Axis(labelAngle=-45)),
        y=alt.Y('Feature:N', title='Feature'),
        color=alt.Color("Kappa:Q", 
                        scale=alt.Scale(domain=[-1, 1], scheme='redyellowgreen'),
                        legend=alt.Legend(title="Cohen's Kappa")
                       ),
        tooltip=['Feature', 'Annotator Pair', alt.Tooltip("Kappa:Q", format='.3f')]
    )
    
    # Add numeric labels to the cells
    # Note: Using alt.datum.Kappa directly for the condition as alt.abs is not a top-level attribute
    text = heatmap.mark_text(baseline='middle').encode(
        text=alt.Text("Kappa:Q", format='.3f'),
        color=alt.condition(
            "(datum.Kappa > 0.5) || (datum.Kappa < -0.5)",
            alt.value('white'),
            alt.value('black')
        )
    )
    
    chart = (heatmap + text).properties(
        title="Heatmap of Pairwise Cohen's Kappa Scores",
        width=600,
        height=300
    )
    
    display(chart)
else:
    print("Pairwise Cohen's Kappa results are not available to plot.")

alt.LayerChart(...)

### 6. Identify and Display Disagreements

To better understand the sources of disagreement, we are using a table that filters for only those items where annotators did not all provide the same annotation for a given feature.

The following code will:
1.  Iterate through each unique item (`ID`) in the dataset.
2.  For each item, check every feature to see if there is more than one unique value (i.e., a disagreement).
3.  Collect all items that have at least one disagreement.
4.  Display a table showing the annotations for these specific items, making it easy to see the differences in ratings.

In [14]:
# Find items with any disagreement
disagreement_ids = []
features = ['all_caps', 'exclamation_marks', 'hedging', 'adjectives', 'unk']

# Group by item ID
grouped = df.groupby('ID')

for name, group in grouped:
    has_disagreement = False
    for feature in features:
        # Make a copy to avoid SettingWithCopyWarning
        feature_series = group[feature].copy()
        
        # Check if the column contains lists and convert them to sorted strings if so
        if feature_series.dropna().apply(lambda x: isinstance(x, list)).any():
            # The lambda function handles non-list values (like NaN) gracefully
            feature_series = feature_series.apply(lambda x: str(sorted(x)) if isinstance(x, list) else x)

        # Check if there's more than one unique value for the feature in this group
        if feature_series.nunique() > 1:
            has_disagreement = True
            break # Move to the next item as soon as one disagreement is found
    if has_disagreement:
        disagreement_ids.append(name)

# Filter the original DataFrame to show only the items with disagreements
disagreement_df = df[df['ID'].isin(disagreement_ids)].copy()

# Optional: Sort for clearer presentation
disagreement_df = disagreement_df.sort_values(by=['ID', 'Annotator'])

# To make disagreements more visible, we can pivot the table
# and display it. This puts annotators in columns.
if not disagreement_df.empty:
    print(f"Found {len(disagreement_ids)} items with at least one disagreement.")
    
    # We can show a pivoted view for each feature to highlight disagreements
    for feature in features:
        # Create a temporary copy for this pivot to handle list-to-string conversion
        temp_df = disagreement_df.copy()
        
        # Convert list to string for pivoting if necessary
        if temp_df[feature].dropna().apply(lambda x: isinstance(x, list)).any():
            temp_df[feature] = temp_df[feature].apply(lambda x: str(sorted(x)) if isinstance(x, list) else x)

        pivot_disagreements = temp_df.pivot_table(
            index=['ID', 'Text'], 
            columns='Annotator', 
            values=feature,
            aggfunc='first' # Use first since there's one value per annotator
        )
        # Filter to show only rows (items) that actually have a disagreement for THIS feature
        feature_disagreement_rows = pivot_disagreements.apply(lambda row: row.nunique() > 1, axis=1)
        
        if feature_disagreement_rows.any():
            print(f"\n--- Disagreements for '{feature}' ---")
            display(pivot_disagreements[feature_disagreement_rows])

else:
    print("No disagreements found across any items.")

Found 740 items with at least one disagreement.

--- Disagreements for 'all_caps' ---


,Annotator,Gemini,Jasmine,Jennifer,Nicole,Rachelle,Shiao-li
ID,Text,,,,,,
6.0,"WTF? » No food, no FEMA: Hurricane Michael’s survivors are furious - The Daily Beast https://apple.news/APp4E5UMtQT2ULJVMPM0ovw …",[],[],['WTF?'],['WTF'],['WTF'],[]
9.0,You know they hate you at work when the TV manager picks you to go out with a camera and operate the LIVE DRIVE car in a tornado.,"['DRIVE', 'LIVE']","['DRIVE', 'LIVE']","['DRIVE', 'LIVE']","['DRIVE', 'LIVE']",['LIVE DRIVE'],"['DRIVE', 'LIVE']"
12.0,President Trump Gives Permission for US Troops to Stay at Trump Hotel in Washington DC https://t.co/PTHILtJqh3 via @gatewaypundit,['DC'],[],[],[],[],[]
14.0,"“The effect of Hurricane Matthew on Haiti is catastrophic"" -food + water crisis... http://ln.is/GSphX by #WSJ via @c0nvey",['WSJ'],[],[],[],[],[]
22.0,"Drove thru NC last week & experienced my first tornado - went right in front of our vehicle, people on the hwy freaked out more than my kids",['NC'],[],[],[],[],[]
...,...,...,...,...,...,...,...
988.0,I say this every hurricane: it's expensive AF to evacuate & not everyone has a reliable vehicle https://twitter.com/cnalive/status/901460592275275777¬†‚Ä¶,['AF'],NaN,NaN,NaN,NaN,[]
989.0,"78-Year-Old Woman, Scared to Drive in Blizzard, Found Dead in Car Outside NJ Burger King http://fb.me/4n2BfuH8G¬†",['NJ'],NaN,NaN,NaN,NaN,[]
992.0,The potential for a border war that has been simmering at the Line of Actual Control (LAC) between India and China could reach a boiling point as hostilities increased on Monday night. https://t.co/5oPBrINKZB,['LAC'],NaN,NaN,NaN,NaN,[]



--- Disagreements for 'exclamation_marks' ---


,Annotator,Gemini,Jasmine,Jennifer,Nicole,Rachelle,Shiao-li
ID,Text,,,,,,
1.0,"Would-be looter in Hurricane Michael-ravaged Florida shot, killed after trying to steal law enforcement vehicle: report\n\nhttps://www.foxnews.com/us/would-be-looter-in-hurricane-michael-ravaged-florida-shot-killed-after-trying-to-steal-law-enforcement-vehicle-report …\n... All looters need to be handled the same way!",['!'],['!'],['!'],[],['!'],['!']
4.0,To all my Texas streamers: PLEASE be safe if the hurricane comes to you! I‚Äôd hate to lose...family...don‚Äôt be a hero. Take shelter. pic.twitter.com/jDPPdMTHwN,['!'],['!'],['!'],[],['!'],['!']
23.0,*eats the medicine @mlp_twilight gives him* whats a solar flare anyways? does the sun explode?!,['!'],['!'],[],['!'],['!'],['!']
48.0,Lately I been stressing make me wanna put a fuck nigga on a stretcher!,['!'],['!'],[],['!'],[],NaN
49.0,Why are so many cars and buses stranded on the highway? Stay off the highway in a blizzard!,['!'],['!'],['!'],['!'],[],NaN
55.0,@jihettly @esd2000 good morning! Yes I'm in the middle of the blizzard. I have plenty of food so I'm happy! Lol. Stay warm.,"['!', '!']","['!', '!']",['! '],[],[],NaN
58.0,"Due to the poor air quality from the wildfires, our teams' Ride to Conquer Cancer was cut short yesterday. Big shout out to the Ski Cellar for still hosting our team for a BBQ yesterday so they could celebrate their training and fundraising successes together! #therideABpic.twitter.com/76gLiXP37q",['!'],['!'],['!'],['!'],[],NaN
59.0,Re: Hurricane Matthew: All of the @ASUTennis Student-Athletes are on their way to safety. Thanks to @ASU_Housing for arranging a safe place!,['!'],['!'],['!'],['!'],[],NaN
73.0,Irony just died a thousand deaths! ???? http://t.co/dBU30ObDxz,['!'],NaN,[],NaN,['!'],NaN



--- Disagreements for 'hedging' ---


,Annotator,Gemini,Jasmine,Jennifer,Nicole,Rachelle,Shiao-li
ID,Text,,,,,,
1.0,"Would-be looter in Hurricane Michael-ravaged Florida shot, killed after trying to steal law enforcement vehicle: report\n\nhttps://www.foxnews.com/us/would-be-looter-in-hurricane-michael-ravaged-florida-shot-killed-after-trying-to-steal-law-enforcement-vehicle-report …\n... All looters need to be handled the same way!","['Would', 'would']",[],[],['Would'],"['Would', 'would']",[]
4.0,To all my Texas streamers: PLEASE be safe if the hurricane comes to you! I‚Äôd hate to lose...family...don‚Äôt be a hero. Take shelter. pic.twitter.com/jDPPdMTHwN,['if'],[],[],[],[],[]
15.0,"i understand everyone has to make a buck, i would encourage you to think if food delivery during a blizzard is the best choice.","['if', 'think', 'would']","['think', 'understand', 'would']","['think', 'understand', 'would']","['think', 'understand', 'would']","['think', 'understand']","['think', 'understand', 'would']"
16.0,"Yeah, but it's also unthinkable to tell hurricane refugees to ""have a great time,"" and to brag about the size of the crowd at shelter.",['about'],[],[],[],['unthinkable'],[]
17.0,Birds in a blizzard. We put out extra sunflower seeds since their usual food sources just got‚Ä¶ https://www.instagram.com/p/BBS2eYWgTbj/¬†,['usual'],[],[],[],[],[]
...,...,...,...,...,...,...,...
987.0,i swea it feels like im about to explode ??,"['about', 'feels', 'like']",NaN,NaN,NaN,NaN,[]
990.0,@spinningbot Are you another Stand-user? If you are I will have to detonate you with my Killer Queen.,['If'],NaN,NaN,NaN,NaN,[]
992.0,The potential for a border war that has been simmering at the Line of Actual Control (LAC) between India and China could reach a boiling point as hostilities increased on Monday night. https://t.co/5oPBrINKZB,['could'],NaN,NaN,NaN,NaN,[]



--- Disagreements for 'adjectives' ---


,Annotator,Gemini,Jasmine,Jennifer,Nicole,Rachelle,Shiao-li
ID,Text,,,,,,
1.0,"Would-be looter in Hurricane Michael-ravaged Florida shot, killed after trying to steal law enforcement vehicle: report\n\nhttps://www.foxnews.com/us/would-be-looter-in-hurricane-michael-ravaged-florida-shot-killed-after-trying-to-steal-law-enforcement-vehicle-report …\n... All looters need to be handled the same way!","['enforcement', 'enforcement', 'trying', 'tryi...",[],[],[],[],[]
2.0,"His argument is correct. Hurricane Rita killed over 100 people on the road, drowned them in their own cars. pic.twitter.com/kulwIZW2EI",['argument'],[],[],[],[],[]
3.0,im praying for all of my friends down in the Caribbean who have no where else to go and are forced to ride through the hurricane. be strong https://twitter.com/ttrogdon/status/905205124699750401¬†‚Ä¶,['praying'],[],[],[],[],[]
5.0,RT @Stacy_Spencer: There is a Tornado warning. Service is canceled. Get to shelter and be safe. ¬´ praying for memphis.,"['praying', 'warning']",[],[],[],[],[]
7.0,"The aftermath of a hurricane is horrific. The heat/humidity is excruciating, no water/ice, no bathing, complete darkness, bugs, no warm food","['bathing', 'darkness', 'excruciating']",['excruciating'],['excruciating'],['excruciating'],"['darkness', 'excruciating']",['excruciating']
...,...,...,...,...,...,...,...
995.0,"Kashmiris and Muslims protesting in Southall London UK after Namaz Juma agaisnt India's violation of Article 370 and called it ""another Palestine alike in the making."" https://t.co/JepImDzd2E","['making', 'protesting']",NaN,NaN,NaN,NaN,[]
996.0,"THR: DESTRUCTIVE TORNADO RIPPING THROUGH TUSCALOOSA, AL NOW! TAKE SHELTER NOW!!! #severe http://dlvr.it/PwRS5 (BN) #tcot",['RIPPING'],NaN,NaN,NaN,NaN,[]
997.0,A two year old video is viral as Indians in Spain now are celebrating building of Ram Mandir in Ayodhya.\nA video is viral on social media in which a group of boys and girls in India attire using Dhols and other Indian musical instruments while walking on.. https://t.co/bWhO3OS4De https://t.co/dtd2CosbPe,"['building', 'celebrating', 'using', 'walking']",NaN,NaN,NaN,NaN,[]



--- Disagreements for 'unk' ---


,Annotator,Gemini,Jasmine,Jennifer,Nicole,Rachelle,Shiao-li
ID,Text,,,,,,
6.0,"WTF? » No food, no FEMA: Hurricane Michael’s survivors are furious - The Daily Beast https://apple.news/APp4E5UMtQT2ULJVMPM0ovw …",['WTF'],['WTF'],['WTF?'],['WTF'],['WTF'],[]
26.0,"Cars melt, power down as wildfire turns California town into ‚Äòburning¬†hell‚Äô https://www.siasat.com/news/cars-melt-power-down-wildfire-turns-california-town-burning-hell-1430954/¬†‚Ä¶pic.twitter.com/MEE6HbXWCs","['hell', 'hell']",[],[],[],[],[]
48.0,Lately I been stressing make me wanna put a fuck nigga on a stretcher!,['fuck'],"['fuck', 'nigga']","['fuck', 'nigga']","['fuck', 'nigga']",[],NaN
68.0,We're ready for the hurricane and Stephen just left like tf are you doing????,[],NaN,['tf'],NaN,['tf'],NaN
80.0,"23 Killed, Cars Melt, Power Down as Wildfire Turns California Town Into ‚ÄòBurning¬†Hell‚Äô https://en.dhwanionline.com/23-killed-cars-melt-power-down-as-wildfire-turns-california-town-into-burning-hell/¬†‚Ä¶","['Hell', 'hell']",NaN,[],NaN,[],NaN
83.0,Grocery employees in DC region are working their asses off to make sure folks have food/supplies ahead of blizzard. MVPs.\n\n#blizzard2016,[],NaN,['asses'],NaN,[],['asses']
89.0,Yay now I get to ride my bike to school through ass crack of hurricane Matthew. I can hear wind howling from inside my house.,['ass'],NaN,['ass'],NaN,[],[]
156.0,Dear food lion fuckers: give me a hard time today and you will be under the blizzard,[],NaN,NaN,NaN,['fuckers'],['fuckers']
186.0,Lmao somewhere along the drainage at work there‚Äôs a clogged pipe. Wash a car & the shop starts to flood.,[],NaN,['Lmao'],NaN,NaN,NaN
